In [ ]:
#!/usr/bin/env python3
"""
Paper-ready (single plot, p75 only):
LCP_p75 vs Third-party domain count using semantic bins + log-y distribution
with per-bin median + p75 markers.

Inputs:
  --csv merged.csv

Outputs:
  --outdir plots_lcp/
    lcp_p75_vs_thirdparty_semantic_bins_logy.png
    lcp_p75_vs_thirdparty_semantic_bins_logy.pdf

Columns used:
  x: browser_metrics.third_party_domains
  y: cwv_mobile.aggregated.LCP_p75
"""

import os
import json
import ast
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# -----------------------------
# Helpers
# -----------------------------
def safe_parse_third_party_count(x):
    """
    Returns an integer count for browser_metrics.third_party_domains.
    Handles:
      - already-numeric
      - list objects
      - JSON strings like '["a.com","b.com"]'
      - python-literal strings like "['a.com', 'b.com']"
      - comma-separated strings
      - empty/NaN
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan

    # If already numeric
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)) and np.isfinite(x):
        return int(x)

    # If already list-like
    if isinstance(x, (list, tuple, set)):
        return len(x)

    # If string-like
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() in {"nan", "none", "null"}:
            return np.nan

        # Try JSON
        try:
            obj = json.loads(s)
            if isinstance(obj, (list, tuple)):
                return len(obj)
            if isinstance(obj, dict):
                return len(obj.keys())
        except Exception:
            pass

        # Try Python literal
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, (list, tuple, set)):
                return len(obj)
            if isinstance(obj, dict):
                return len(obj.keys())
        except Exception:
            pass

        # Fallback: comma-separated
        if "," in s:
            parts = [p.strip() for p in s.split(",") if p.strip()]
            return len(parts)

    return np.nan


def make_semantic_bin(v):
    """
    Semantic bins:
      0, 1-2, 3-5, 6-10, 11+
    """
    if not np.isfinite(v):
        return None
    v = int(v)
    if v == 0:
        return "0"
    if 1 <= v <= 2:
        return "1–2"
    if 3 <= v <= 5:
        return "3–5"
    if 6 <= v <= 10:
        return "6–10"
    return "11+"


def violin_with_markers_seaborn(
    ax,
    out_df,
    order,
    ylabel,
    title,
    ymin=100,
    ymax=None,
    palette_map=None,
    use_grid=True
):
    """
    Seaborn violinplot (per-bin colors) + median & p75 markers + n labels.
    """
    # Violin (seaborn)
    sns.violinplot(
        data=out_df,
        x="tp_bin",
        y="lcp_p75",
        order=order,
        palette=palette_map,
        cut=0,
        inner=None,
        linewidth=0,
        saturation=1,
        ax=ax
    )

    # Slight transparency for bodies (keep edges off)
    for coll in ax.collections:
        try:
            coll.set_alpha(0.28)
        except Exception:
            pass

    # Add median + p75 markers + n labels (bigger, readable)
    for i, lbl in enumerate(order):
        vals = out_df.loc[out_df["tp_bin"] == lbl, "lcp_p75"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        med = np.median(vals)
        p75 = np.percentile(vals, 75)

        ax.scatter(i, med, s=55, marker="o", color="#111111", zorder=5)
        ax.scatter(i, p75, s=70, marker="^", color="#111111", zorder=5)

        y_annot = max(p75, med) * 1.12
        ax.text(
            i,
            y_annot,
            f"n={len(vals)}",
            ha="center",
            va="bottom",
            fontsize=13,
            alpha=0.90
        )

    ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    # Gridlines ONLY if helpful (y-only; light)
    if use_grid:
        ax.grid(True, which="major", axis="y", alpha=0.22, linewidth=0.7)
        ax.grid(False, which="minor", axis="y")
        ax.set_axisbelow(True)
    else:
        ax.grid(False)

    ax.set_ylim(bottom=ymin)
    if ymax is not None:
        ax.set_ylim(top=ymax)

    # Legend (custom handles)
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker="o", color="#111111", linestyle="None", markersize=7, label="Median (per bin)"),
        Line2D([0], [0], marker="^", color="#111111", linestyle="None", markersize=8, label="P75 (per bin)"),
    ]
    ax.legend(handles=handles, loc="upper left", frameon=True)


# -----------------------------
# Main
# -----------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True, help="Path to merged.csv")
    ap.add_argument("--outdir", default="plots_lcp", help="Output directory")
    ap.add_argument("--ymin", type=float, default=100.0, help="Lower y-limit (ms) for log scale (e.g., 100)")
    ap.add_argument("--trim_y_pct", type=float, default=None,
                    help="Optional: trim extreme high LCP_p75 by percentile (e.g., 99.5). Keeps low values.")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    # --- Seaborn theme + BIG fonts (paper-ready) ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 22,
        "axes.labelsize": 20,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 14,
        "font.size": 16,
        "axes.titlepad": 12,
        "axes.labelpad": 12,
    })

    df = pd.read_csv(args.csv)

    # Columns (p75 only)
    x_col = "browser_metrics.third_party_domains"
    y_col = "cwv_mobile.aggregated.LCP_p75"

    missing = [c for c in [x_col, y_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in CSV: {missing}")

    # Parse third-party count (handles list/string/numeric)
    x_count = df[x_col].apply(safe_parse_third_party_count)

    # Build semantic bins
    bins = x_count.apply(make_semantic_bin)

    # Numeric LCP p75
    lcp_p75 = pd.to_numeric(df[y_col], errors="coerce")

    # Combine and drop NaNs
    out = pd.DataFrame({
        "tp_bin": bins,
        "lcp_p75": lcp_p75
    }).dropna(subset=["tp_bin", "lcp_p75"])

    # Optional: trim only extreme high tail for readability
    if args.trim_y_pct is not None:
        hi = np.nanpercentile(out["lcp_p75"], args.trim_y_pct)
        out = out[out["lcp_p75"] <= hi]

    order = ["0", "1–2", "3–5", "6–10", "11+"]

    # --- Contrasting palette (ColorHunt-style hexes; per-bin) ---
    # You can swap these with any ColorHunt palette later without touching logic.
    colors = ["#222831", "#00ADB5", "#F8B500", "#FF6F61", "#6A67CE"]
    palette_map = {lbl: colors[i] for i, lbl in enumerate(order)}

    # Single plot (DPI=1200)
    fig, ax = plt.subplots(figsize=(10.2, 5.9), dpi=1200, constrained_layout=True)

    violin_with_markers_seaborn(
        ax,
        out_df=out,
        order=order,
        ylabel=r"$\rightarrow$ LCP p75 (ms, log scale)",
        title="LCP_p75 vs third-party domain count (semantic bins; log-y)",
        ymin=args.ymin,
        palette_map=palette_map,
        use_grid=True  # log-y distribution benefits from light y-grid
    )

    # X label with arrow (LaTeX-style mathtext)
    ax.set_xlabel(r"Third-party domain count (semantic bins) $\rightarrow$")

    # Suptitle (bigger, clean)
    n_total = len(out)
    fig.suptitle(f"Third-party domains vs LCP_p75 (n={n_total})", y=1.03, fontsize=18)

    # Clean spines
    sns.despine(ax=ax)

    png_path = os.path.join(args.outdir, "lcp_p75_vs_thirdparty_semantic_bins_logy.png")
    pdf_path = os.path.join(args.outdir, "lcp_p75_vs_thirdparty_semantic_bins_logy.pdf")
    fig.savefig(png_path, dpi=1200, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=1200, bbox_inches="tight")
    plt.close(fig)

    print(png_path)
    print(pdf_path)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# plot_lcp_vs_num_stylesheets_binned.py
"""
Semantic-binned LCP_p75 vs #stylesheets with LOG y-axis (paper-ready)

Semantic bins for stylesheets:
  0, 1, 2, 3–4, 5–7, 8+

Uses ONLY:
  x: browser_metrics.num_stylesheets
  y: cwv_mobile.aggregated.LCP_p75

Produces ONE plot:
- plot_lcp_p75_vs_num_stylesheets_semantic_logy.png

Plot shows (per x-bin):
- ONE line: P75(LCP_p75) with bootstrap 95% CI error bars.
"""

import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def semantic_bin_stylesheets(x: pd.Series) -> pd.Categorical:
    """Map num_stylesheets -> semantic bins: 0, 1, 2, 3–4, 5–7, 8+"""
    x = pd.to_numeric(x, errors="coerce")

    labels = ["0", "1", "2", "3–4", "5–7", "8+"]
    b = pd.Series(pd.NA, index=x.index, dtype="object")

    b[x == 0] = "0"
    b[x == 1] = "1"
    b[x == 2] = "2"
    b[(x >= 3) & (x <= 4)] = "3–4"
    b[(x >= 5) & (x <= 7)] = "5–7"
    b[x >= 8] = "8+"

    return pd.Categorical(b, categories=labels, ordered=True)


def bootstrap_ci(values: np.ndarray, stat_fn, boots: int, seed: int) -> tuple[float, float]:
    """Bootstrap 95% CI for a statistic. Returns (lo, hi)."""
    rng = np.random.default_rng(seed)
    n = len(values)
    if n <= 1:
        return (np.nan, np.nan)

    stats = np.empty(boots, dtype=float)
    for i in range(boots):
        sample = values[rng.integers(0, n, size=n)]
        stats[i] = stat_fn(sample)

    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(lo), float(hi)


def summarize_bins_p75only(df: pd.DataFrame, bin_col: str, y_col: str, boots: int, seed: int) -> pd.DataFrame:
    """
    For each x-bin, compute:
      - n
      - p75(y) + bootstrap CI
    """
    rows = []
    for cat in df[bin_col].cat.categories:
        sub = df[df[bin_col] == cat][y_col].dropna().to_numpy(dtype=float)
        n = len(sub)
        if n == 0:
            rows.append(dict(xbin=cat, n=0, y_p75=np.nan, y_p75_lo=np.nan, y_p75_hi=np.nan))
            continue

        p75 = float(np.percentile(sub, 75))
        p75_lo, p75_hi = bootstrap_ci(
            sub,
            lambda a: np.percentile(a, 75),
            boots=boots,
            seed=seed + (hash((cat, y_col, "p75")) % 10_000),
        )

        rows.append(dict(xbin=cat, n=n, y_p75=p75, y_p75_lo=p75_lo, y_p75_hi=p75_hi))

    out = pd.DataFrame(rows)
    out["x"] = np.arange(len(out), dtype=float)
    return out


def make_plot(summary: pd.DataFrame, title: str, xlabel: str, ylabel: str, outpath: Path,
              ymin: float | None, ymax: float | None) -> None:
    """Single figure: ONE line (p75) with bootstrap CI error bars. LOG y-axis."""
    # --- Seaborn + paper-ready typography + DPI=1200 ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 22,
        "axes.labelsize": 20,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 14,
        "font.size": 16,
        "axes.titlepad": 12,
        "axes.labelpad": 12,
    })

    fig, ax = plt.subplots(figsize=(11, 5), dpi=1200)

    y = summary["y_p75"].to_numpy(float)
    yerr_lo = (summary["y_p75"] - summary["y_p75_lo"]).to_numpy(float)
    yerr_hi = (summary["y_p75_hi"] - summary["y_p75"]).to_numpy(float)

    # --- Contrasting color (single-line plot), clean styling ---
    line_color = "#00ADB5"  # high-contrast teal (swapable)
    ax.errorbar(
        summary["x"], y, yerr=np.vstack([yerr_lo, yerr_hi]),
        marker="^",
        markersize=8,
        markerfacecolor=line_color,
        markeredgecolor="white",
        markeredgewidth=0.8,
        color=line_color,
        linewidth=2.4,
        capsize=4,
        label="P75(LCP_p75) per bin",
        zorder=3
    )

    # n labels (bigger)
    for xi, n in zip(summary["x"], summary["n"]):
        if n > 0:
            yy = summary.loc[summary["x"] == xi, "y_p75"].values[0]
            ax.annotate(
                f"n={int(n)}",
                (xi, yy),
                textcoords="offset points",
                xytext=(0, 8),
                ha="center",
                fontsize=13,
                alpha=0.90
            )

    # --- Title & axis labels (with arrows via mathtext/LaTeX-style) ---
    ax.set_title(title)
    ax.set_xlabel(rf"{xlabel} $\rightarrow$")
    ax.set_ylabel(rf"$\rightarrow$ {ylabel} (ms, log scale)")

    ax.set_xticks(summary["x"])
    ax.set_xticklabels(summary["xbin"].tolist())

    ax.set_yscale("log")
    if ymin is not None or ymax is not None:
        ax.set_ylim(bottom=ymin, top=ymax)

    # Gridlines only if helpful: log-y benefits from light major y-grid
    ax.grid(True, which="major", axis="y", alpha=0.22, linewidth=0.7)
    ax.grid(False, which="minor", axis="y")
    ax.set_axisbelow(True)

    ax.legend(loc="best", frameon=True)

    sns.despine(ax=ax)

    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(outpath, dpi=1200, bbox_inches="tight")
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="merge_1000.csv", help="Path to CSV (default: merge_1000.csv)")
    ap.add_argument("--outdir", default="plots_lcp", help="Output directory")
    ap.add_argument("--boots", type=int, default=1000, help="Bootstrap samples per bin")
    ap.add_argument("--seed", type=int, default=7, help="RNG seed")
    ap.add_argument("--ymin", type=float, default=None, help="Optional y-axis min (ms) (log-scale)")
    ap.add_argument("--ymax", type=float, default=None, help="Optional y-axis max (ms) (log-scale)")
    args = ap.parse_args()

    csv_path = Path(args.csv)
    outdir = Path(args.outdir)

    x_col = "browser_metrics.num_stylesheets"
    y_col = "cwv_mobile.aggregated.LCP_p75"

    df = pd.read_csv(csv_path)

    if x_col not in df.columns:
        raise KeyError(f"Missing column: {x_col}")
    if y_col not in df.columns:
        raise KeyError(f"Missing column: {y_col}")

    df["_xbin"] = semantic_bin_stylesheets(df[x_col])

    d = df[["_xbin", y_col]].copy()
    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    d = d.dropna(subset=["_xbin", y_col])

    summary = summarize_bins_p75only(d, bin_col="_xbin", y_col=y_col, boots=args.boots, seed=args.seed)

    title = "LCP_p75 vs Number of stylesheets (semantic bins; log y; bootstrap CI)"
    xlabel = "Number of stylesheets (semantic bins)"
    ylabel = "LCP_p75"

    outpath = outdir / "plot_lcp_p75_vs_num_stylesheets_semantic_logy.png"
    make_plot(summary, title, xlabel, ylabel, outpath, ymin=args.ymin, ymax=args.ymax)

    print(outpath)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# plot_lcp_vs_num_images_logx_lowess.py
"""
LCP_p75 vs Number of images (log-x scatter + LOWESS) with optional x/y trim.

- x: browser_metrics.num_images (optionally trim to x <= percentile via --trim_x)
- x-axis plotted as log1p(num_images)
- y: cwv_mobile.aggregated.LCP_p75 (ms), linear
- ONE scatter + ONE LOWESS trend
- optional y-trim (by percentile) to reduce extreme tail dominance

Example:
  python plot_lcp_vs_num_images_logx_lowess.py --csv merged_1000.csv --outdir plots_lcp --trim_x 0.95 --trim_y 0.99
"""

import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess


def safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def lowess_curve(x: np.ndarray, y: np.ndarray, frac: float) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]
    sm = lowess(y_sorted, x_sorted, frac=frac, it=2, return_sorted=True)
    return sm[:, 0], sm[:, 1]


def should_use_grid(x_vals: np.ndarray, y_vals: np.ndarray) -> bool:
    """
    Gridlines only if genuinely helpful.
    Heuristic: enable if y-range is wide enough that reading values benefits.
    """
    y_vals = y_vals[np.isfinite(y_vals)]
    if len(y_vals) < 10:
        return False
    y_min, y_max = np.min(y_vals), np.max(y_vals)
    if y_min <= 0:
        return True
    # If dynamic range is large, light y-grid helps.
    return (y_max / y_min) >= 3.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--outdir", default="plots_lcp")
    ap.add_argument("--x_col", default="browser_metrics.num_images")
    ap.add_argument("--lcp_p75_col", default="cwv_mobile.aggregated.LCP_p75")
    ap.add_argument("--lowess_frac", type=float, default=0.35)
    ap.add_argument("--trim_x", type=float, default=None,
                    help="Keep rows with x <= this percentile (e.g., 0.95).")
    ap.add_argument("--trim_y", type=float, default=None,
                    help="Keep rows with y <= this percentile (e.g., 0.99).")
    ap.add_argument("--min_y", type=float, default=0.0)
    ap.add_argument("--dpi", type=int, default=1200)  # force 1200
    ap.add_argument("--grid", type=str, default="auto",
                    help="Gridlines: 'off' | 'on' | 'auto' (default).")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    # --- Seaborn + paper-ready typography + DPI=1200 ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 22,
        "axes.labelsize": 20,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 14,
        "font.size": 16,
        "axes.titlepad": 12,
        "axes.labelpad": 12,
    })

    df = pd.read_csv(args.csv)

    if args.x_col not in df.columns:
        raise SystemExit(f"Missing column: {args.x_col}")
    if args.lcp_p75_col not in df.columns:
        raise SystemExit(f"Missing column: {args.lcp_p75_col}")

    x = safe_numeric(df[args.x_col])
    y = safe_numeric(df[args.lcp_p75_col])

    keep = x.notna() & y.notna()
    df2 = pd.DataFrame({"x": x[keep], "y": y[keep]}).copy()
    df2 = df2[(df2["x"] >= 0) & (df2["y"] >= 0)]

    # ----- Trim on X (by percentile) -----
    x_thr = None
    if args.trim_x is not None:
        if not (0 < args.trim_x <= 1):
            raise SystemExit("--trim_x must be in (0,1], e.g., 0.97")
        x_thr = df2["x"].quantile(args.trim_x)
        df2 = df2[df2["x"] <= x_thr]

    # ----- Optional trim on Y (by percentile) -----
    y_thr = None
    if args.trim_y is not None:
        if not (0 < args.trim_y <= 1):
            raise SystemExit("--trim_y must be in (0,1], e.g., 0.99")
        y_thr = df2["y"].quantile(args.trim_y)
        df2 = df2[df2["y"] <= y_thr]

    n = len(df2)
    if n < 10:
        raise SystemExit(f"Not enough points after filtering (n={n}). Reduce trimming or check column names.")

    x_log = np.log1p(df2["x"].to_numpy())
    y_np = df2["y"].to_numpy()

    x_s, y_s = lowess_curve(x_log, y_np, frac=args.lowess_frac)

    # ---- Plot ----
    fig, ax = plt.subplots(figsize=(9.2, 5.4), dpi=1200)

    # Strong contrasting colors (single series + trend)
    scatter_color = "#6A67CE"  # purple
    line_color = "#F8B500"     # gold

    ax.scatter(
        x_log, y_np,
        s=34, alpha=0.38,
        marker="^",
        color=scatter_color,
        edgecolor="white",
        linewidth=0.5,
        label="LCP p75 (raw)",
        zorder=2
    )
    ax.plot(
        x_s, y_s,
        linewidth=3.0,
        color=line_color,
        label=rf"P75 LOWESS (frac={args.lowess_frac})",
        zorder=3
    )

    # Axis labels with arrows (mathtext/LaTeX-style)
    ax.set_xlabel(r"$\log(1 + \#\ \mathrm{images}) \rightarrow$")
    ax.set_ylabel(r"$\rightarrow$ LCP p75 (ms)")

    ax.set_ylim(bottom=args.min_y)

    # Tick labels in original image-count units (but positioned at log1p)
    x_max = df2["x"].max()
    candidates = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
    tick_counts = [t for t in candidates if t <= x_max]
    if len(tick_counts) == 0:
        tick_counts = [0, int(x_max)]
    elif tick_counts[-1] != int(x_max):
        tick_counts = tick_counts + [int(x_max)]
    tick_locs = [np.log1p(t) for t in tick_counts]
    ax.set_xticks(tick_locs)
    ax.set_xticklabels([str(int(t)) for t in tick_counts])

    title = f"LCP p75 vs Number of images (log-x + LOWESS), n={n}"
    if x_thr is not None:
        title += f", x≤P{int(args.trim_x*100)}={x_thr:.1f}"
    if y_thr is not None:
        title += f", y≤P{int(args.trim_y*100)}={y_thr:.0f}ms"
    ax.set_title(title)

    ax.legend(frameon=True, framealpha=0.95, loc="upper left")

    # --- Gridlines: ONLY if needed ---
    grid_mode = args.grid.lower().strip()
    if grid_mode == "on":
        ax.grid(True, which="major", alpha=0.18, linewidth=0.7)
    elif grid_mode == "off":
        ax.grid(False)
    else:  # auto
        use_grid = should_use_grid(x_log, y_np)
        if use_grid:
            ax.grid(True, which="major", axis="y", alpha=0.18, linewidth=0.7)
        else:
            ax.grid(False)

    sns.despine(ax=ax)

    outpath = os.path.join(args.outdir, "plot_lcp_p75_vs_num_images_logx_lowess_trimx.png")
    fig.tight_layout()
    fig.savefig(outpath, dpi=1200, bbox_inches="tight")
    plt.close(fig)

    print(outpath)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
"""
Single-plot (trimmed) : LCP p75 vs DOMContentLoaded with log-x + LOWESS

Uses YOUR columns:
  x:  browser_metrics.resource_timing.domContentLoaded
  y:  cwv_mobile.aggregated.LCP_p75

Behavior:
- Cleans non-numeric / inf / NaN
- Keeps x>=0, y>=0
- Trims to bulk only using percentiles (defaults: 97/97)
- Plots ONE scatter + ONE LOWESS curve
- Saves: plots_lcp/plot_lcp_p75_vs_domcontentloaded_lowess_trimmed.png
"""

import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

XCOL = "browser_metrics.resource_timing.domContentLoaded"
YCOL = "cwv_mobile.aggregated.LCP_p75"


def to_numeric(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    return s.replace([np.inf, -np.inf], np.nan)


def lowess_curve(x: np.ndarray, y: np.ndarray, frac: float = 0.35, it: int = 2):
    order = np.argsort(x)
    xs = x[order]
    ys = y[order]
    sm = lowess(ys, xs, frac=frac, it=it, return_sorted=True)
    return sm[:, 0], sm[:, 1]


def should_use_grid(y_vals: np.ndarray) -> bool:
    """
    Gridlines only if genuinely helpful.
    Heuristic: enable light y-grid if y-range is wide enough to aid reading.
    """
    y_vals = y_vals[np.isfinite(y_vals)]
    if len(y_vals) < 10:
        return False
    y_min, y_max = np.min(y_vals), np.max(y_vals)
    if y_min <= 0:
        return True
    return (y_max / y_min) >= 3.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True, help="Input CSV, e.g., merged.csv")
    ap.add_argument("--outdir", default="plots_lcp", help="Output directory")
    ap.add_argument("--trim_x_pct", type=float, default=97.0, help="Trim x percentile (keep x <= P). Use 0 to disable.")
    ap.add_argument("--trim_y_pct", type=float, default=97.0, help="Trim y percentile (keep y <= P). Use 0 to disable.")
    ap.add_argument("--lowess_frac", type=float, default=0.35, help="LOWESS smoothing (0.25–0.5 typical)")
    ap.add_argument("--lowess_it", type=int, default=2, help="LOWESS robust iterations")
    ap.add_argument("--alpha", type=float, default=0.35, help="Scatter alpha")
    ap.add_argument("--s", type=float, default=28, help="Scatter size")
    ap.add_argument("--dpi", type=int, default=1200, help="Output DPI (forced default 1200)")
    ap.add_argument("--grid", type=str, default="auto", help="Gridlines: 'off' | 'on' | 'auto' (default)")
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)

    # --- Seaborn + paper-ready typography + DPI=1200 ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 22,
        "axes.labelsize": 20,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 14,
        "font.size": 16,
        "axes.titlepad": 12,
        "axes.labelpad": 12,
    })

    df = pd.read_csv(args.csv)

    # Validate columns early
    for c in [XCOL, YCOL]:
        if c not in df.columns:
            candidates = [cc for cc in df.columns if ("LCP" in cc) or ("domContentLoaded" in cc)]
            hint = "\n".join(candidates[:60]) if candidates else "(no similar columns found)"
            raise SystemExit(f"Missing column: {c}\nCandidates (first 60):\n{hint}")

    # Load + clean
    x = to_numeric(df[XCOL])
    y = to_numeric(df[YCOL])

    data = pd.DataFrame({"x": x, "y": y}).dropna()
    data = data[(data["x"] >= 0) & (data["y"] >= 0)]

    if len(data) < 10:
        raise SystemExit(f"Too few valid rows after cleaning: n={len(data)}")

    # Trim thresholds (computed on ORIGINAL x/y space)
    x_thr = None
    y_thr = None
    if args.trim_x_pct and args.trim_x_pct > 0:
        x_thr = np.percentile(data["x"].to_numpy(), args.trim_x_pct)
    if args.trim_y_pct and args.trim_y_pct > 0:
        y_thr = np.percentile(data["y"].to_numpy(), args.trim_y_pct)

    mask = np.ones(len(data), dtype=bool)
    if x_thr is not None:
        mask &= data["x"].to_numpy() <= x_thr
    if y_thr is not None:
        mask &= data["y"].to_numpy() <= y_thr

    bulk = data[mask].copy()

    if len(bulk) < 10:
        raise SystemExit(
            f"Too few points after trimming: n={len(bulk)}. "
            f"Try higher percentiles (e.g., 99/99) or disable one trim with 0."
        )

    # Log-x for plotting + LOWESS
    bulk["x_log"] = np.log1p(bulk["x"].to_numpy())

    xb = bulk["x_log"].to_numpy()
    yb = bulk["y"].to_numpy()

    x_curve, y_curve = lowess_curve(xb, yb, frac=args.lowess_frac, it=args.lowess_it)

    # Plot (single) — strong contrasting colors
    fig, ax = plt.subplots(figsize=(8.8, 5.6), dpi=1200, constrained_layout=True)
    scatter_color = "#00ADB5"  # teal
    line_color = "#F8B500"     # gold

    ax.scatter(
        xb, yb,
        alpha=args.alpha,
        s=args.s,
        marker="^",
        color=scatter_color,
        edgecolor="white",
        linewidth=0.5,
        label="LCP p75 (trimmed raw)",
        zorder=2
    )
    ax.plot(
        x_curve, y_curve,
        linewidth=3.0,
        color=line_color,
        label=rf"P75 LOWESS (frac={args.lowess_frac})",
        zorder=3
    )

    subtitle = f"n={len(bulk)}/{len(data)}"
    if x_thr is not None:
        subtitle += f", x≤P{int(args.trim_x_pct)}={x_thr:.1f}ms"
    if y_thr is not None:
        subtitle += f", y≤P{int(args.trim_y_pct)}={y_thr:.0f}ms"

    ax.set_title(f"LCP p75 vs DOMContentLoaded (log-x + LOWESS), {subtitle}")

    # Axis labels with arrows (mathtext/LaTeX-style)
    ax.set_xlabel(r"$\log(1 + \mathrm{DOMContentLoaded\ [ms]}) \rightarrow$")
    ax.set_ylabel(r"$\rightarrow$ LCP p75 (ms)")

    ax.legend(frameon=True, framealpha=0.95, loc="upper left")

    # Gridlines: ONLY if needed (or forced)
    grid_mode = args.grid.lower().strip()
    if grid_mode == "on":
        ax.grid(True, which="major", axis="y", alpha=0.18, linewidth=0.7)
    elif grid_mode == "off":
        ax.grid(False)
    else:  # auto
        if should_use_grid(yb):
            ax.grid(True, which="major", axis="y", alpha=0.18, linewidth=0.7)
        else:
            ax.grid(False)

    sns.despine(ax=ax)

    outpath = os.path.join(args.outdir, "plot_lcp_p75_vs_domcontentloaded_lowess_trimmed.png")
    fig.savefig(outpath, dpi=1200, bbox_inches="tight")
    print(outpath)


if __name__ == "__main__":
    main()


# plot_lcp_vs_domcontentloaded_lowess.py --csv merged_1000.csv --outdir plots_lcp --trim_x_pct 97 --trim_y_pct 95 --lowess_frac 0.35 --lowess_it 1


In [ ]:
#!/usr/bin/env python3
"""
Replot (trimmed only): LCP (p75) vs Total transfer size (bytes)
- x-axis: log1p(total_transfer_size)
- y-axis: LCP p75 in ms (linear)
- scatter for LCP_p75 (trimmed; outliers removed)
- LOWESS trend line on trimmed data only

Input CSV must contain:
  browser_metrics.total_transfer_size
  cwv_mobile.aggregated.LCP_p75
"""

import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def lowess_np(x, y, frac=0.35, it=2):
    """
    LOWESS via statsmodels if available, else a simple rolling median fallback.
    Returns smoothed (xs, ys) sorted by x.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 5:
        order = np.argsort(x)
        return x[order], y[order]

    order = np.argsort(x)
    x_s, y_s = x[order], y[order]

    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        sm = lowess(y_s, x_s, frac=frac, it=it, return_sorted=True)
        return sm[:, 0], sm[:, 1]
    except Exception:
        # Fallback: rolling median on x-sorted points
        w = max(5, int(frac * len(x_s)))
        if w % 2 == 0:
            w += 1
        y_med = (
            pd.Series(y_s)
            .rolling(window=w, center=True, min_periods=max(3, w // 3))
            .median()
            .to_numpy()
        )
        # fill edges
        mask = np.isfinite(y_med)
        if mask.any():
            first = np.argmax(mask)
            last = len(y_med) - 1 - np.argmax(mask[::-1])
            y_med[:first] = y_med[first]
            y_med[last + 1:] = y_med[last]
        else:
            y_med = y_s
        return x_s, y_med


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="merged.csv")
    ap.add_argument("--outdir", default="plots_lcp")
    ap.add_argument("--frac", type=float, default=0.35, help="LOWESS smoothing fraction")
    ap.add_argument("--it", type=int, default=2, help="LOWESS robust iterations")
    ap.add_argument("--trim_x_p", type=float, default=95.0,
                    help="Trim percentile for x (total_transfer_size). Keep x <= P. 0 disables.")
    ap.add_argument("--trim_y_p", type=float, default=97.0,
                    help="Trim percentile for y (LCP p75). Keep y <= P. 0 disables.")
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--dpi", type=int, default=1200)  # enforce 1200
    args = ap.parse_args()

    np.random.seed(args.seed)
    os.makedirs(args.outdir, exist_ok=True)

    # --- Seaborn + paper-ready typography + DPI=1200 ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 22,
        "axes.labelsize": 20,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "legend.fontsize": 14,
        "font.size": 16,
        "axes.titlepad": 12,
        "axes.labelpad": 12,
    })

    x_col = "browser_metrics.total_transfer_size"
    y_col = "cwv_mobile.aggregated.LCP_p75"

    df = pd.read_csv(args.csv)

    # Coerce numeric
    for c in [x_col, y_col]:
        if c not in df.columns:
            candidates = [cc for cc in df.columns if ("LCP" in cc) or ("transfer" in cc)]
            hint = "\n".join(candidates[:50]) if candidates else "(no similar columns found)"
            raise SystemExit(
                f"Missing column: {c}\n"
                f"Columns containing 'LCP' or 'transfer' (first 50):\n{hint}"
            )
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Keep rows with valid x and y
    d = df[[x_col, y_col]].copy()
    d = d[np.isfinite(d[x_col]) & (d[x_col] >= 0)]
    d = d[np.isfinite(d[y_col])]

    if len(d) == 0:
        raise SystemExit("No valid rows after filtering finite x/y.")

    # Compute thresholds in original (non-log) spaces
    x_thr = None
    if args.trim_x_p and args.trim_x_p > 0:
        x_thr = np.nanpercentile(d[x_col].to_numpy(), args.trim_x_p)

    y_thr = None
    if args.trim_y_p and args.trim_y_p > 0:
        y_thr = np.nanpercentile(d[y_col].to_numpy(), args.trim_y_p)

    # Apply trimming (bulk only)
    mask = np.ones(len(d), dtype=bool)
    if x_thr is not None:
        mask &= d[x_col].to_numpy() <= x_thr
    if y_thr is not None:
        mask &= d[y_col].to_numpy() <= y_thr

    d_trim = d[mask].copy()

    if len(d_trim) == 0:
        raise SystemExit(
            "All points were trimmed away. Try increasing trim percentiles "
            "(e.g., --trim_x_p 99 --trim_y_p 99) or disable one with 0."
        )

    # Plot (trimmed only)
    fig, ax = plt.subplots(figsize=(10.8, 6.2), dpi=1200)
    x = np.log1p(d_trim[x_col].to_numpy())
    y = d_trim[y_col].to_numpy()

    # Contrasting colors
    scatter_color = "#6A67CE"  # purple
    line_color = "#F8B500"     # gold

    ax.scatter(
        x, y,
        s=34, alpha=0.42,
        marker="^",
        color=scatter_color,
        edgecolor="white",
        linewidth=0.5,
        label="LCP p75 (trimmed raw)",
        zorder=2
    )

    if len(y) >= 5:
        xs, ys = lowess_np(x, y, frac=args.frac, it=args.it)
        ax.plot(
            xs, ys,
            linewidth=3.0,
            color=line_color,
            label=rf"P75 LOWESS (frac={args.frac})",
            zorder=3
        )

    # --- NO GRIDLINES (as requested) ---
    ax.grid(False)

    # Axis labels with arrows (mathtext/LaTeX-style)
    ax.set_xlabel(r"$\log(1 + \mathrm{total\ transfer\ size\ [bytes]}) \rightarrow$")
    ax.set_ylabel(r"$\rightarrow$ LCP p75 (ms)")

    suffix = f" (trimmed: n={len(d_trim)}/{len(d)}"
    if x_thr is not None:
        suffix += f", x≤P{args.trim_x_p}={x_thr:.0f}B"
    if y_thr is not None:
        suffix += f", y≤P{args.trim_y_p}={y_thr:.0f}ms"
    suffix += ")"
    ax.set_title("LCP p75 vs Total transfer size (log-x + LOWESS)" + suffix)

    ax.legend(frameon=True, framealpha=0.95, loc="upper left")

    sns.despine(ax=ax)

    out = os.path.join(args.outdir, "plot_lcp_p75_total_transfer_size_lowess_trimmed.png")
    fig.tight_layout()
    fig.savefig(out, dpi=1200, bbox_inches="tight")
    plt.close(fig)
    print(out)


if __name__ == "__main__":
    main()


# python plot_lcp_total_transfer_size_lowess_1000.py --csv merged_1000.csv --outdir plots_lcp --trim_x_p 99 --trim_y_p 99

In [ ]:
#!/usr/bin/env python3
# plot_lcp_box_third_party.py
"""
Single Plot: Boxplot of LCP_p75 by third-party domain count (semantic bins) with log-y.

Input:
  --csv merged_1000.csv

Output:
  plots_lcp/simple_box_lcp_p75_vs_third_party.png

Run:
  python plot_lcp_box_third_party.py --csv merged_1000.csv --outdir plots_lcp
"""

from __future__ import annotations

import argparse
import ast
import textwrap
from pathlib import Path
from typing import Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def try_parse_listlike(x) -> Optional[List]:
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                v = ast.literal_eval(s)
                return v if isinstance(v, list) else None
            except Exception:
                return None
    return None


def get_third_party_count(df: pd.DataFrame) -> pd.Series:
    # Prefer explicit count columns if present
    for col in ["browser_metrics.third_party_domain_count", "third_party_domain_count"]:
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce")

    # Else derive from third_party_domains list
    if "browser_metrics.third_party_domains" in df.columns:
        out = []
        for v in df["browser_metrics.third_party_domains"].values:
            lst = try_parse_listlike(v)
            if lst is None:
                out.append(np.nan)
            else:
                clean = [str(d).strip() for d in lst if str(d).strip()]
                out.append(len(set(clean)))
        return pd.Series(out, index=df.index, dtype="float")

    raise KeyError(
        "No third-party domain count column found. "
        "Expected one of: browser_metrics.third_party_domain_count, third_party_domain_count, "
        "or browser_metrics.third_party_domains."
    )


def tp_bin(n: float) -> Optional[str]:
    if not np.isfinite(n):
        return None
    n = int(n)
    if n == 0:
        return "0"
    if 1 <= n <= 2:
        return "1–2"
    if 3 <= n <= 5:
        return "3–5"
    if 6 <= n <= 10:
        return "6–10"
    return "11+"


def make_boxplot(df: pd.DataFrame, ycol: str, title: str, outpath: Path):
    # --- Seaborn + paper-ready typography + DPI=1200 ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 18,
        "axes.labelsize": 18,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 13,
        "font.size": 14,
        "axes.titlepad": 16,
        "axes.labelpad": 12,
    })

    bin_order = ["0", "1–2", "3–5", "6–10", "11+"]

    data = [df.loc[df["tp_bin"] == b, ycol].dropna().values for b in bin_order]

    fig, ax = plt.subplots(figsize=(9.5, 4.8), dpi=1200)

    # Contrasting palette (ColorHunt-ish)
    colors = ["#222831", "#00ADB5", "#F8B500", "#FF6F61", "#6A67CE"]

    bp = ax.boxplot(
        data,
        labels=bin_order,
        showfliers=False,
        patch_artist=True,
        widths=0.7
    )

    # Style boxes
    for i, box in enumerate(bp["boxes"]):
        box.set_facecolor(colors[i])
        box.set_alpha(0.35)
        box.set_edgecolor("none")

    # Style medians/whiskers/caps
    for med in bp["medians"]:
        med.set_color("#111111")
        med.set_linewidth(2.0)
    for w in bp["whiskers"]:
        w.set_color("#111111")
        w.set_linewidth(1.3)
    for c in bp["caps"]:
        c.set_color("#111111")
        c.set_linewidth(1.3)

    ax.set_yscale("log")

    # Labels with arrows
    ax.set_ylabel(r"$\rightarrow$ LCP p75 (ms, log scale)")
    ax.set_xlabel(r"Third-party domain count (semantic bins) $\rightarrow$")

    # Wrap title
    wrapped_title = "\n".join(textwrap.wrap(title, width=52))
    ax.set_title(wrapped_title, pad=18)

    # --- NO GRIDLINES ---
    ax.grid(False)

    # --- Place n-labels INSIDE plot (data coords) to avoid title overlap ---
    # Choose a stable y-position inside the axes, near upper region.
    ymin, ymax = ax.get_ylim()
    y_text = ymax / 1.7  # safely inside for log-scale
    for i, b in enumerate(bin_order, start=1):
        n = len(df.loc[df["tp_bin"] == b, ycol].dropna())
        ax.text(
            i, y_text, f"n={n}",
            ha="center", va="center",
            fontsize=12, alpha=0.90,
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="none", alpha=0.75)
        )

    sns.despine(ax=ax)

    # Give extra headroom for 2-line title (prevents clipping)
    fig.subplots_adjust(top=0.84)

    fig.savefig(outpath, dpi=1200, bbox_inches="tight")
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="merged_1000.csv")
    ap.add_argument("--outdir", default="plots_lcp")
    ap.add_argument("--min_lcp_ms", type=float, default=1.0)
    args = ap.parse_args()

    df = pd.read_csv(args.csv)
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    lcp_p75 = "cwv_mobile.aggregated.LCP_p75"
    if lcp_p75 not in df.columns:
        raise KeyError(f"Missing column: {lcp_p75}")

    df = df.copy()
    df["third_party_domain_count"] = get_third_party_count(df)
    df["tp_bin"] = df["third_party_domain_count"].apply(tp_bin)

    df[lcp_p75] = pd.to_numeric(df[lcp_p75], errors="coerce")

    df = df[df["tp_bin"].notna() & (df[lcp_p75] > args.min_lcp_ms)]

    outpath = outdir / "simple_box_lcp_p75_vs_third_party.png"
    make_boxplot(
        df, lcp_p75,
        "LCP_p75 vs Third-party domain count (boxplot, log-y; outliers hidden)",
        outpath
    )

    print(outpath)


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
import os
import json
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _safe_json_loads(x):
    if pd.isna(x):
        return None
    if isinstance(x, (list, dict)):
        return x
    s = str(x).strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        return None

def compute_largest_resource_size(df: pd.DataFrame, col="browser_metrics.large_resources") -> pd.Series:
    largest = []
    for v in df.get(col, pd.Series([None]*len(df))):
        arr = _safe_json_loads(v)
        if isinstance(arr, list) and len(arr) > 0:
            sizes = []
            for item in arr:
                if isinstance(item, dict):
                    sz = item.get("size", None)
                    if isinstance(sz, (int, float)) and np.isfinite(sz):
                        sizes.append(float(sz))
            largest.append(max(sizes) if sizes else np.nan)
        else:
            largest.append(np.nan)
    return pd.Series(largest, index=df.index)

def quantile_binned_two_stats(x_log: pd.Series, y_med: pd.Series, y_p75: pd.Series, n_bins=8):
    d = pd.DataFrame({"x": x_log, "y_med": y_med, "y_p75": y_p75}).dropna()
    d = d[np.isfinite(d["x"]) & np.isfinite(d["y_med"]) & np.isfinite(d["y_p75"])]

    if len(d) < max(10, n_bins * 3):
        n_bins = max(3, min(n_bins, len(d) // 3))

    d["bin"] = pd.qcut(d["x"], q=n_bins, duplicates="drop")
    grp = d.groupby("bin", observed=True)

    rows = []
    for b, g in grp:
        left_log = float(b.left)
        right_log = float(b.right)
        # convert back to bytes for labels
        left_b = float(np.expm1(left_log))
        right_b = float(np.expm1(right_log))

        rows.append({
            "left_b": left_b,
            "right_b": right_b,
            "center_log": (left_log + right_log) / 2.0,
            "n": int(len(g)),
            "lcp_median_bin": float(np.median(g["y_med"].to_numpy())),
            "lcp_p75_bin": float(np.median(g["y_p75"].to_numpy())),  # robust: median of per-page p75s in bin
        })

    out = pd.DataFrame(rows).sort_values("center_log").reset_index(drop=True)

    def fmt_bytes(v):
        if v >= 1024**2:
            return f"{v/(1024**2):.1f}MB"
        if v >= 1024:
            return f"{v/1024:.0f}KB"
        return f"{v:.0f}B"

    out["bin_label"] = out.apply(lambda r: f"{fmt_bytes(r['left_b'])}–{fmt_bytes(r['right_b'])}", axis=1)
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="merged.csv")
    ap.add_argument("--outdir", default="plots_lcp")
    ap.add_argument("--bins", type=int, default=8)
    args = ap.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    df = pd.read_csv(args.csv, low_memory=False)

    largest_bytes = compute_largest_resource_size(df)
    x_log = np.log1p(largest_bytes)

    y_med = pd.to_numeric(df.get("cwv_mobile.aggregated.LCP_median"), errors="coerce")
    y_p75 = pd.to_numeric(df.get("cwv_mobile.aggregated.LCP_p75"), errors="coerce")

    S = quantile_binned_two_stats(x_log, y_med, y_p75, n_bins=args.bins)

    x = np.arange(len(S))

    # nice, simple, colorblind-friendly colors
    c1 = "#1f77b4"  # blue
    c2 = "#ff7f0e"  # orange

    plt.figure(figsize=(10.5, 4.8), dpi=200)
    plt.title("LCP vs Largest resource size (quantile-binned x)", fontsize=12, fontweight="bold")

    plt.plot(x, S["lcp_median_bin"].values, "-o", color=c1, label="LCP median (per bin)", linewidth=2, markersize=5)
    plt.plot(x, S["lcp_p75_bin"].values, "-s", color=c2, label="LCP p75 (per bin)", linewidth=2, markersize=5)

    # annotate n
    for i, n in enumerate(S["n"].values):
        y_top = max(S["lcp_median_bin"].iloc[i], S["lcp_p75_bin"].iloc[i])
        plt.text(i, y_top * 1.03, f"n={n}", ha="center", va="bottom", fontsize=8, alpha=0.8)

    plt.xticks(x, S["bin_label"].tolist(), rotation=25, ha="right", fontsize=8)
    plt.ylabel("LCP (ms)")
    plt.xlabel("Largest resource size bin (bytes) (quantile bins)")
    plt.grid(True, alpha=0.18)
    plt.legend(frameon=True, fontsize=9)
    plt.ylim(bottom=1000)
    plt.tight_layout()

    outpath = os.path.join(args.outdir, "lcp_vs_largest_resource_ONEPLOT.png")
    plt.savefig(outpath, bbox_inches="tight")
    plt.close()
    print(f"[OK] Saved: {outpath}")

if __name__ == "__main__":
    main()
#python plot_lcp_box_third_party.py  --csv merged_1000.csv --outdir plots_lcp

In [ ]:
#!/usr/bin/env python3
"""
INP vs DOM processing time (resource_timing.dom) using Quantile Regression (tau=0.5, 0.75)
SINGLE PLOT (bulk only; outliers removed):
- Drops outliers by keeping: x <= P95 and y <= P99 (y based on max(INP_median, INP_p75))
- Plots INP_median and INP_p75 as raw scatter + quantreg trend lines (τ=0.50, 0.75) fit on bulk only
- DPI=1200, NO gridlines by default

Usage:
  python plot_inp_vs_dom_timing_quantreg.py --csv merged.csv --outdir plots_inp --drop_x_zeros --drop_inp_zeros
"""

import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm


def safe_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def fit_quantreg(x: np.ndarray, y: np.ndarray, tau: float):
    X = sm.add_constant(x)
    mod = sm.QuantReg(y, X)
    return mod.fit(q=tau)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="merged.csv")
    ap.add_argument("--outdir", default="plots_inp")

    ap.add_argument("--x_col", default="browser_metrics.resource_timing.dom")
    ap.add_argument("--inp_median_col", default="cwv_mobile.aggregated.INP_median")
    ap.add_argument("--inp_p75_col", default="cwv_mobile.aggregated.INP_p75")

    ap.add_argument("--drop_x_zeros", action="store_true", help="Drop rows where x == 0")
    ap.add_argument("--drop_inp_zeros", action="store_true", help="Drop rows where INP_median==0 OR INP_p75==0")

    ap.add_argument("--x_clip_pct", type=float, default=95.0, help="Keep x <= P (bulk x threshold)")
    ap.add_argument("--y_clip_pct", type=float, default=99.0, help="Keep y <= P (bulk y threshold)")
    ap.add_argument("--seed", type=int, default=7)

    ap.add_argument("--alpha", type=float, default=0.55)
    ap.add_argument("--ms", type=float, default=28.0)
    ap.add_argument("--dpi", type=int, default=1200)
    ap.add_argument("--grid", action="store_true", help="Enable gridlines (default: off)")
    args = ap.parse_args()

    np.random.seed(args.seed)
    os.makedirs(args.outdir, exist_ok=True)

    # --- Seaborn + typography ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 18,
        "axes.labelsize": 18,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 12,
        "font.size": 14,
        "axes.titlepad": 12,
        "axes.labelpad": 10,
    })

    df = pd.read_csv(args.csv)

    # --- Extract + coerce numeric ---
    if args.x_col not in df.columns:
        raise SystemExit(f"Missing column: {args.x_col}")
    if args.inp_median_col not in df.columns:
        raise SystemExit(f"Missing column: {args.inp_median_col}")
    if args.inp_p75_col not in df.columns:
        raise SystemExit(f"Missing column: {args.inp_p75_col}")

    x = safe_numeric(df[args.x_col])
    y_med = safe_numeric(df[args.inp_median_col])
    y_p75 = safe_numeric(df[args.inp_p75_col])

    d = pd.DataFrame({"x": x, "y_med": y_med, "y_p75": y_p75}).dropna()
    d = d[d["x"] >= 0]

    if args.drop_x_zeros:
        d = d[d["x"] != 0]
    if args.drop_inp_zeros:
        d = d[(d["y_med"] != 0) & (d["y_p75"] != 0)]

    if len(d) < 10:
        raise SystemExit(f"Not enough rows after cleaning: n={len(d)}")

    # --- Bulk thresholds (outlier removal) ---
    x_thr = np.percentile(d["x"].to_numpy(), args.x_clip_pct)
    y_max = np.maximum(d["y_med"].to_numpy(), d["y_p75"].to_numpy())
    y_thr = np.percentile(y_max, args.y_clip_pct)

    bulk = d[(d["x"] <= x_thr) & (y_max <= y_thr)].copy()

    if len(bulk) < 10:
        raise SystemExit(
            f"Too few points after outlier removal: n={len(bulk)}. "
            f"Try higher percentiles: --x_clip_pct 99 --y_clip_pct 99"
        )

    # --- Fit quantile regressions on BULK only ---
    x_b = bulk["x"].to_numpy()
    yb_med = bulk["y_med"].to_numpy()
    yb_p75 = bulk["y_p75"].to_numpy()

    taus = [0.50, 0.75]
    fits = {}
    for tau in taus:
        fits[("med", tau)] = fit_quantreg(x_b, yb_med, tau)
        fits[("p75", tau)] = fit_quantreg(x_b, yb_p75, tau)

    # --- Grid for lines ---
    x_grid = np.linspace(x_b.min(), x_b.max(), 240)
    Xg = sm.add_constant(x_grid)

    # --- Plot (single) ---
    fig, ax = plt.subplots(figsize=(10.8, 6.2), dpi=1200)

    # Contrasting colors
    c_med = "#00ADB5"      # teal
    c_p75 = "#6A67CE"      # purple
    c_med_q = "#F8B500"    # gold
    c_p75_q = "#FF6F61"    # coral

    ax.scatter(
        bulk["x"], bulk["y_med"],
        s=args.ms, alpha=args.alpha,
        color=c_med, edgecolor="white", linewidth=0.4,
        label="INP_median (raw; outliers removed)"
    )
    ax.scatter(
        bulk["x"], bulk["y_p75"],
        s=args.ms, alpha=args.alpha,
        marker="^",
        color=c_p75, edgecolor="white", linewidth=0.4,
        label="INP_p75 (raw; outliers removed)"
    )

    # QuantReg lines
    ax.plot(
        x_grid, fits[("med", 0.50)].predict(Xg),
        linewidth=3.0, color=c_med_q,
        label="QuantReg τ=0.50 on INP_median"
    )
    ax.plot(
        x_grid, fits[("med", 0.75)].predict(Xg),
        linewidth=3.0, color=c_med_q, linestyle="--", alpha=0.9,
        label="QuantReg τ=0.75 on INP_median"
    )

    ax.plot(
        x_grid, fits[("p75", 0.50)].predict(Xg),
        linewidth=3.0, color=c_p75_q,
        label="QuantReg τ=0.50 on INP_p75"
    )
    ax.plot(
        x_grid, fits[("p75", 0.75)].predict(Xg),
        linewidth=3.0, color=c_p75_q, linestyle="--", alpha=0.9,
        label="QuantReg τ=0.75 on INP_p75"
    )

    ax.set_xlabel(rf"{args.x_col} (ms) $\rightarrow$")
    ax.set_ylabel(r"$\rightarrow$ INP (ms)")

    ax.set_title(
        f"INP vs DOM processing time (QuantReg τ=0.50 & 0.75)\n"
        f"Outliers removed: x≤P{args.x_clip_pct:g}={x_thr:.1f}ms, "
        f"y≤P{args.y_clip_pct:g}={y_thr:.1f}ms, n={len(bulk)}/{len(d)}",
        pad=14
    )

    if args.grid:
        ax.grid(True, alpha=0.18)
    else:
        ax.grid(False)

    ax.legend(loc="upper left", frameon=True, framealpha=0.95)
    sns.despine(ax=ax)

    fig.tight_layout()
    outpath = os.path.join(args.outdir, "plot_inp_vs_dom_timing_quantreg_bulk_only.png")
    fig.savefig(outpath, dpi=1200, bbox_inches="tight")
    plt.close(fig)
    print(outpath)


if __name__ == "__main__":
    main()

# python plot_inp_vs_dom_timing_quantreg.py --csv merged.csv --outdir plots_inp --x_clip_pct 95 --y_clip_pct 99 --drop_x_zeros --drop_inp_zeros


In [ ]:
#!/usr/bin/env python3
"""
INP vs Total transfer size (log-x) with quantile-regression trend lines (BULK ONLY; outliers removed).

- Reads merged.csv
- Uses x = log1p(browser_metrics.total_transfer_size)
- Keeps BULK only: x_log <= Ptrim_x_pct AND max(INP_median, INP_p75) <= Ptrim_y_pct
- Plots INP_median + INP_p75 as raw points (bulk only)
- Fits QuantReg lines for tau=0.50 and tau=0.75 on bulk only
- SINGLE PLOT (no tail panel)
- DPI=1200, gridlines OFF by default

Output:
  plots_inp/plot_inp_vs_total_transfer_size_logx_quantreg_bulk_only.png
"""

import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm


def quantile_fit_sorted(x, y, tau):
    """Quantile regression y ~ a + b*x. Returns xs(sorted), yhat(xs), fitted result."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 10:
        raise ValueError("Too few points to fit quantile regression.")
    order = np.argsort(x)
    xs = x[order]
    ys = y[order]
    X = sm.add_constant(xs)
    model = sm.QuantReg(ys, X)
    res = model.fit(q=tau)
    yhat = res.predict(X)
    return xs, yhat, res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="merged.csv")
    ap.add_argument("--outdir", default="plots_inp")

    ap.add_argument("--trim_x_pct", type=float, default=95.0, help="Bulk x cutoff percentile on x_log")
    ap.add_argument("--trim_y_pct", type=float, default=99.0, help="Bulk y cutoff percentile on max(INP_median, INP_p75)")

    ap.add_argument("--drop_x_zeros", action="store_true", help="Drop rows where total_transfer_size <= 0")
    ap.add_argument("--drop_inp_zeros", action="store_true", help="Drop rows where INP_median<=0 OR INP_p75<=0")

    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--alpha", type=float, default=0.55)
    ap.add_argument("--ms", type=float, default=28.0)
    ap.add_argument("--dpi", type=int, default=1200)
    ap.add_argument("--grid", action="store_true", help="Enable gridlines (default OFF)")
    args = ap.parse_args()

    np.random.seed(args.seed)
    os.makedirs(args.outdir, exist_ok=True)

    # Columns (as per your schema)
    x_col = "browser_metrics.total_transfer_size"
    y_med_col = "cwv_mobile.aggregated.INP_median"
    y_p75_col = "cwv_mobile.aggregated.INP_p75"

    df = pd.read_csv(args.csv)

    # Validate columns
    for c in [x_col, y_med_col, y_p75_col]:
        if c not in df.columns:
            raise SystemExit(f"Missing column: {c}")

    # Coerce numeric
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[y_med_col] = pd.to_numeric(df[y_med_col], errors="coerce")
    df[y_p75_col] = pd.to_numeric(df[y_p75_col], errors="coerce")

    # Base validity
    d = df[[x_col, y_med_col, y_p75_col]].dropna().copy()

    if args.drop_x_zeros:
        d = d[d[x_col] > 0]
    if args.drop_inp_zeros:
        d = d[(d[y_med_col] > 0) & (d[y_p75_col] > 0)]

    if len(d) < 10:
        raise SystemExit(f"Too few usable rows after filtering: n={len(d)}")

    # log-x
    d["x_log"] = np.log1p(d[x_col].to_numpy())

    # Bulk region thresholds
    y_max = np.maximum(d[y_med_col].to_numpy(), d[y_p75_col].to_numpy())
    x_thr = np.percentile(d["x_log"].to_numpy(), args.trim_x_pct)
    y_thr = np.percentile(y_max, args.trim_y_pct)

    bulk = d[(d["x_log"] <= x_thr) & (y_max <= y_thr)].copy()

    if len(bulk) < 10:
        raise SystemExit(
            f"Too few points after outlier removal: n={len(bulk)}/{len(d)}. "
            f"Try higher percentiles: --trim_x_pct 99 --trim_y_pct 99"
        )

    # Quantile regression on bulk (separately for INP_median and INP_p75)
    xs_m50, yhat_m50, _ = quantile_fit_sorted(bulk["x_log"], bulk[y_med_col], tau=0.50)
    xs_m75, yhat_m75, _ = quantile_fit_sorted(bulk["x_log"], bulk[y_med_col], tau=0.75)

    xs_p50, yhat_p50, _ = quantile_fit_sorted(bulk["x_log"], bulk[y_p75_col], tau=0.50)
    xs_p75, yhat_p75, _ = quantile_fit_sorted(bulk["x_log"], bulk[y_p75_col], tau=0.75)

    # --- Styling (DPI=1200 + larger fonts, no overlap) ---
    sns.set_theme(style="white", context="talk")
    plt.rcParams.update({
        "figure.dpi": 1200,
        "savefig.dpi": 1200,
        "axes.titlesize": 18,
        "axes.labelsize": 18,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 12,
        "font.size": 14,
        "axes.titlepad": 12,
        "axes.labelpad": 10,
    })

    # Contrasting colors (consistent)
    c_med = "#00ADB5"      # teal (median points)
    c_p75 = "#6A67CE"      # purple (p75 points)
    c_med_q = "#F8B500"    # gold (median lines)
    c_p75_q = "#FF6F61"    # coral (p75 lines)

    # ---- Plot (single bulk-only) ----
    fig, ax = plt.subplots(figsize=(10.8, 6.2), dpi=1200)

    ax.scatter(
        bulk["x_log"], bulk[y_med_col],
        s=args.ms, alpha=args.alpha,
        color=c_med, edgecolor="white", linewidth=0.4,
        label="INP_median (raw; outliers removed)"
    )
    ax.scatter(
        bulk["x_log"], bulk[y_p75_col],
        s=args.ms, alpha=args.alpha,
        marker="^",
        color=c_p75, edgecolor="white", linewidth=0.4,
        label="INP_p75 (raw; outliers removed)"
    )

    # Trend lines (QuantReg)
    ax.plot(xs_m50, yhat_m50, linewidth=3.0, color=c_med_q, label="QuantReg τ=0.50 on INP_median")
    ax.plot(xs_m75, yhat_m75, linewidth=3.0, color=c_med_q, linestyle="--", alpha=0.9,
            label="QuantReg τ=0.75 on INP_median")

    ax.plot(xs_p50, yhat_p50, linewidth=3.0, color=c_p75_q, label="QuantReg τ=0.50 on INP_p75")
    ax.plot(xs_p75, yhat_p75, linewidth=3.0, color=c_p75_q, linestyle="--", alpha=0.9,
            label="QuantReg τ=0.75 on INP_p75")

    ax.set_xlabel(rf"log(1 + total_transfer_size bytes) $\rightarrow$  ({x_col})")
    ax.set_ylabel(r"$\rightarrow$ INP (ms)")

    ax.set_title(
        "INP vs Total transfer size (log-x): Quantile trends (median & p75)\n"
        f"Outliers removed: x_log≤P{args.trim_x_pct:.0f}={x_thr:.3f}, "
        f"y≤P{args.trim_y_pct:.0f}={y_thr:.1f}ms, n={len(bulk)}/{len(d)}",
        pad=14
    )

    if args.grid:
        ax.grid(True, alpha=0.18)
    else:
        ax.grid(False)

    ax.legend(loc="upper left", frameon=True, framealpha=0.95)
    sns.despine(ax=ax)

    fig.tight_layout()
    outpath = os.path.join(args.outdir, "plot_inp_vs_total_transfer_size_logx_quantreg_bulk_only.png")
    fig.savefig(outpath, dpi=1200, bbox_inches="tight")
    plt.close(fig)

    print(outpath)


if __name__ == "__main__":
    main()

#python plot_inp_total_transfer_size.py --csv merged.csv --outdir plots_inp --drop_x_zeros --drop_inp_zeros